# Лабораторная №7 — сегментация Oxford-IIIT Pets

Все пять этапов методички: загрузка архива images/annotations; предобработка, визуализация и заполнение пропусков; архитектура сегментатора; обучение; результаты. Маски trimaps имеют **3 класса** (фон, животное, граница), поэтому используем корректную трёхклассовую softmax-сегментацию.

## 1. Загрузка данных Oxford-IIIT Pets

Ссылки из методички перенесены на HTTPS. Загрузка архивов может занять несколько минут.

In [ ]:
import os, pathlib, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from sklearn.model_selection import train_test_split
tf.keras.utils.set_random_seed(42)
!wget -q -c https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz -O images.tar.gz
!wget -q -c https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz -O annotations.tar.gz
!tar -xf images.tar.gz
!tar -xf annotations.tar.gz
print("Файлы:",len(list(pathlib.Path("images").glob("*.jpg"))))

## 2. Предобработка — сопоставление изображений и масок по имени

В методичке пути сортируются по отдельности, что может нарушить соответствие. Здесь пары строятся по stem имени файла. Ресайз 200×200, как в PDF; для экономии памяти используется ограничение количества изображений.

In [ ]:
input_dir=pathlib.Path("images")
target_dir=pathlib.Path("annotations/trimaps")
pairs=[(p,target_dir/(p.stem+".png")) for p in sorted(input_dir.glob("*.jpg")) if (target_dir/(p.stem+".png")).exists()]
assert pairs,"Нет пар изображений и масок"
print("Пар:",len(pairs))
IMG_SIZE=(200,200)
MAX_IMAGES=2000  # None — весь набор, может требовать больше RAM
if MAX_IMAGES is not None:
    rng=np.random.default_rng(42)
    ix=rng.choice(len(pairs),size=min(MAX_IMAGES,len(pairs)),replace=False)
    pairs=[pairs[i] for i in sorted(ix)]
def read_pair(pair):
    with Image.open(pair[0]) as im:
        x=np.asarray(im.convert("RGB").resize(IMG_SIZE,Image.Resampling.BILINEAR),dtype=np.float32)/255.
    with Image.open(pair[1]) as im:
        # trimaps labels are 1,2,3; transform to 0,1,2
        y=np.asarray(im.convert("L").resize(IMG_SIZE,Image.Resampling.NEAREST),dtype=np.int32)-1
    assert y.min()>=0 and y.max()<3,(pair,y.min(),y.max())
    return x,y
X,Y=zip(*(read_pair(pair) for pair in pairs))
X=np.stack(X);Y=np.stack(Y)
print(X.shape,Y.shape,np.unique(Y))
X_train,X_val,Y_train,Y_val=train_test_split(X,Y,test_size=0.2,random_state=42)

### 2.1. Вывод исходных изображений и масок

In [ ]:
fig,axs=plt.subplots(2,6,figsize=(16,6))
for i in range(6):
    axs[0,i].imshow(X_train[i]);axs[1,i].imshow(Y_train[i],vmin=0,vmax=2,cmap="viridis")
    axs[0,i].axis("off");axs[1,i].axis("off")
plt.tight_layout();plt.show()

## 3. Топология нейронной сети — заполненные пропуски

Свёрточный encoder–decoder с residual skip-соединениями. Выход — три вероятности Softmax на каждый пиксель; loss — sparse categorical crossentropy.

In [ ]:
def conv_block(x,filters):
    x=layers.Conv2D(filters,3,padding="same",activation="relu")(x)
    return layers.Conv2D(filters,3,padding="same",activation="relu")(x)
inputs=keras.Input((200,200,3))
c1=conv_block(inputs,16)
p1=layers.MaxPooling2D(2)(c1)
c2=conv_block(p1,32)
p2=layers.MaxPooling2D(2)(c2)
b=conv_block(p2,64)
u2=layers.UpSampling2D(2)(b)
u2=conv_block(layers.Concatenate()([u2,c2]),32)
u1=layers.UpSampling2D(2)(u2)
u1=conv_block(layers.Concatenate()([u1,c1]),16)
out=layers.Conv2D(3,1,activation="softmax")(u1)
model=keras.Model(inputs,out)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model.summary()

## 4. Обучение — допишите код

In [ ]:
history=model.fit(X_train,Y_train,validation_data=(X_val,Y_val),batch_size=8,epochs=10,verbose=1)
plt.plot(history.history["loss"],label="train")
plt.plot(history.history["val_loss"],label="validation")
plt.xlabel("Эпоха");plt.ylabel("Ошибка");plt.legend();plt.show()

## 5. Результат работы — предсказание и визуализация

In [ ]:
pred=model.predict(X_val[:6],verbose=0).argmax(axis=-1)
fig,axs=plt.subplots(3,6,figsize=(15,8))
for i in range(6):
    axs[0,i].imshow(X_val[i]);axs[1,i].imshow(Y_val[i],vmin=0,vmax=2);axs[2,i].imshow(pred[i],vmin=0,vmax=2)
    for r in range(3):axs[r,i].axis("off")
plt.tight_layout();plt.show()
print("Validation metrics:",model.evaluate(X_val,Y_val,verbose=0,return_dict=True))

## Вывод

Оцените качество сегментации животного, фона и границ по фактическим результатам. Пропуски учебной архитектуры и обучения заполнены; результат заранее не подставляется.